# 1. Prepare the reviews

Google reviews of 14 luxury beauty points of sale in Paris, 2016 to 2026.

### Table of contents
1.1 Setup  
1.2 How the reviews were collected  
1.3 Load the reviews  
1.4 Keep beauty reviews only  
1.5 Store groups  
1.6 Language groups  
1.7 Reviews that name a staff member  
1.8 Save

## 1.1 Setup

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 20)
sns.set_style('whitegrid')

## 1.2 How the reviews were collected

- Tool: Apify actor `kaix/google-maps-reviews-scraper`, run on 22 November 2026.
- Settings: all reviews published since 1 January 2016, newest first, English translation added by Google, region FR.
- Reviewer names, profiles and photos were **not kept**. Owner replies are reduced to a yes/no flag.
- Department stores sell far more than beauty, so for them only reviews found with beauty keywords were collected: perfume, parfum, fragrance, makeup, maquillage, cosmetics, beauty, beauté, 香水.
- The raw file stays private (Google terms and privacy). Only aggregated tables are published.

## 1.3 Load the reviews

In [2]:
df = pd.read_csv('../data_private/reviews_raw.csv')
print(df.shape)
df.head(3)[['place_name', 'publishedAt', 'language', 'rating']]

(18695, 11)


,place_name,publishedAt,language,rating
0,Maison Francis Kurkdjian,2026-11-11T03:25:46.298Z,ru,5
1,Maison Francis Kurkdjian,2026-11-06T20:34:06.484Z,en,5
2,Maison Francis Kurkdjian,2026-11-04T11:26:40.413Z,en,5


In [3]:
# English text: Google translation when the review is not in English
df['text_en'] = df['textTranslated'].fillna(df['text'])
df['date'] = pd.to_datetime(df['publishedAt'], utc=True).dt.tz_localize(None)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.to_period('M').dt.to_timestamp()
df['has_text'] = df['text_en'].notna()
df = df[df['year'] >= 2016]
df['has_text'].value_counts()

has_text
True     13793
False     4902
Name: count, dtype: int64

## 1.4 Keep beauty reviews only

Guerlain, Chanel, Maison Francis Kurkdjian, Sephora and the CDG duty-free shops are beauty stores, so all their reviews are kept.  
Dior 30 Montaigne and the department stores also sell fashion, food and more, so only reviews that talk about beauty are kept.

In [4]:
beauty_words = r'perfume|fragrance|parfum|scent|cologne|make-?up|cosmetic|beauty|skin ?care|lipstick|foundation|serum|sephora'
mixed_stores = ['DIOR Paris 30 Montaigne', 'Galeries Lafayette Haussmann', 'Printemps Haussmann',
                'Le Bon Marché', 'Samaritaine']

is_mixed = df['place_name'].isin(mixed_stores)
is_beauty = df['text_en'].str.contains(beauty_words, case=False, na=False)
df = df[~is_mixed | is_beauty].copy()
df['place_name'].value_counts()

place_name
Sephora                                            9799
Guerlain                                           1736
Extime Duty Free Paris | Paris-CDG - Terminal 1    1655
Printemps Haussmann                                1011
Galeries Lafayette Haussmann                        579
Chanel                                              545
Maison Francis Kurkdjian                            470
Samaritaine                                         275
Extime Duty Free Paris                              236
Le Bon Marché                                       181
Printemps Maison Beauté Enfant                      134
DIOR Paris 30 Montaigne                              84
Name: count, dtype: int64

## 1.5 Store groups

In [5]:
store_names = {
    'ChIJs1v-d8Rv5kcRsWeOHxtHs00': 'Chanel, Champs-Élysées',
    'ChIJibU2rQNu5kcR3z3WvzoFkrw': 'Chanel, Marais',
    'ChIJRXuGDS5u5kcRGvGB5LiygG4': 'Kurkdjian, rue d\'Alger',
    'ChIJBTyr4gJu5kcRQWnsCx7fARs': 'Kurkdjian, Marais',
    'ChIJjTGLk8Nv5kcRe67t4KMp1CE': 'Guerlain, Champs-Élysées',
    'ChIJ6WA7DsVv5kcRphanoQC_FWQ': 'Dior, 30 Montaigne',
    'ChIJF_ehBDhv5kcR1eAGXQiPV20': 'Sephora, Champs-Élysées',
    'ChIJixChAzdu5kcRkxWrzUU4d-Y': 'Galeries Lafayette',
    'ChIJKwI_HgVv5kcRA-FR6d7yQ_Q': 'Printemps Haussmann',
    'ChIJE93fQ6Fv5kcRW9fMVIMxHj0': 'Printemps Maison Beauté',
    'ChIJAYa7ntNx5kcRcmJxXPZ7m9k': 'Le Bon Marché',
    'ChIJ7fyHuSFu5kcRbJXo82kbNjU': 'Samaritaine',
    'ChIJq43tSB4-5kcRLDxnjLUgX3U': 'CDG Terminal 1 duty free',
    'ChIJLzR6f_8V5kcRPxHpYQiltYE': 'CDG Terminal 3 duty free'}
store_group = {'Chanel': 'Brand boutique', 'Kurkdjian': 'Brand boutique', 'Guerlain': 'Brand boutique',
               'Dior': 'Brand boutique', 'Sephora': 'Sephora', 'CDG': 'Airport duty free'}

df['store'] = df['place_id'].map(store_names)
df['store_group'] = df['store'].str.split(',| ').str[0].map(store_group).fillna('Department store')
df.groupby('store_group')['store'].value_counts()

store_group        store                   
Airport duty free  CDG Terminal 1 duty free    1655
                   CDG Terminal 3 duty free     236
Brand boutique     Guerlain, Champs-Élysées    1736
                   Chanel, Champs-Élysées       323
                   Kurkdjian, rue d'Alger       283
                   Chanel, Marais               222
                   Kurkdjian, Marais            187
                   Dior, 30 Montaigne            84
Department store   Printemps Haussmann         1011
                   Galeries Lafayette           579
                   Samaritaine                  275
                   Le Bon Marché                181
                   Printemps Maison Beauté      134
Sephora            Sephora, Champs-Élysées     9799
Name: count, dtype: int64

## 1.6 Language groups

Google does not show where a reviewer comes from. The language of the review is used instead:
- **French**: mostly local shoppers
- **English**: visitors and English-speaking residents
- **Other languages**: almost always international visitors

In [6]:
df['language_group'] = np.select([df['language'].eq('fr'), df['language'].eq('en')],
                                 ['French', 'English'], 'Other languages')
df.loc[~df['has_text'], 'language_group'] = 'No text'
df['language_group'].value_counts()

language_group
French             8524
No text            4203
English            2624
Other languages    1354
Name: count, dtype: int64

## 1.7 Reviews that name a staff member

Many 5-star reviews thank a named sales associate. Stores often ask happy customers to leave a review, so these reviews push ratings up.  
They are flagged so results can be shown with and without them. The keyword rules catch most of them but not all, so the share is a minimum.

In [7]:
not_names = {'You', 'The', 'Them', 'Her', 'Him', 'God', 'All', 'Everyone', 'Team', 'Sephora', 'Dior', 'Guerlain', 'Chanel',
             'Paris', 'It', 'She', 'He', 'They', 'This', 'Everything', 'Service', 'Staff', 'Store', 'Shop', 'We', 'My', 'Our',
             'Maison', 'Francis', 'Galeries', 'Lafayette', 'Printemps', 'Samaritaine', 'Extime', 'Thank', 'Thanks', 'Very', 'So', 'Everybody', 'Somebody', 'Nobody', 'Someone', 'Security', 'Management', 'Manager', 'Customer', 'Lady', 'Gentleman', 'Girl', 'Man', 'Woman', 'Saleswoman', 'Salesman', 'Seller', 'Advisor', 'Assistant', 'Employee', 'Person', 'Who', 'That', 'There', 'Which', 'Nothing', 'Nobody', 'Someone', 'Anyone'}
name_patterns = [
    r'(?:[Tt]hanks? (?:you )?(?:so much |very much )?(?:to )?|special thanks to |shout ?out to |assisted by |helped by |served by |advised by )([A-Z][a-zà-ÿ]+)',
    r'\b([A-Z][a-zà-ÿ]+) (?:was|is) (?:so |very |extremely |incredibly |super |really |truly )?(?:helpful|amazing|wonderful|great|lovely|kind|patient|fantastic|excellent|professional|attentive|sweet|friendly|adorable|charming)',
    r'\b([A-Z][a-zà-ÿ]{2,}) (?:gave|helped|took|welcomed|advised|recommended|guided|assisted|served)\b']

def names_staff(text):
    if not isinstance(text, str):
        return False
    found = [m for p in name_patterns for m in re.findall(p, text)]
    return any(n not in not_names for n in found)

df['staff_named'] = df['text_en'].apply(names_staff)
df.groupby('store_group')['staff_named'].mean().round(2)

store_group
Airport duty free    0.27
Brand boutique       0.08
Department store     0.20
Sephora              0.17
Name: staff_named, dtype: float64

## 1.8 Save

In [8]:
cols = ['reviewId', 'store', 'store_group', 'date', 'year', 'month', 'language', 'language_group', 'rating',
        'has_text', 'staff_named', 'owner_replied', 'text_en']
df[cols].to_csv('../data_private/reviews_clean.csv', index=False)
print(len(df), 'reviews saved')

16705 reviews saved
